# DASH: Diversified Aggregation of SHAP
## Authoritative Benchmark Notebook (v7)

> **Supersedes** `demo_benchmark_6.ipynb`. All expensive computations are
> checkpoint-cached to disk. On re-run, completed sections load in seconds.
> Use `clear_all_checkpoints()` or `clear_checkpoint(name)` to force re-computation.

**Caraker, Arnold, Rhoads (2026)**

This notebook produces all results, figures, and tables for the paper
*"First-Mover Bias in Gradient Boosting Explanations: Mechanism, Detection,
and Resolution."*

Figures are saved to `results/figures/` in both PDF and PNG formats for
direct inclusion in the LaTeX draft.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings, time, sys, platform, pickle, os, json

# Record environment
print(f'Python:      {sys.version}')
print(f'Platform:    {platform.platform()}')
for pkg_name, pkg in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib),
                       ('seaborn', sns)]:
    print(f'{pkg_name:12s} {pkg.__version__}')
import xgboost, shap, sklearn, scipy
for pkg_name, pkg in [('xgboost', xgboost), ('shap', shap),
                       ('scikit-learn', sklearn), ('scipy', scipy)]:
    print(f'{pkg_name:12s} {pkg.__version__}')

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='shap')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10

from dash.core.pipeline import DASHPipeline
from dash.core.consensus import compute_consensus
from dash.core.filtering import performance_filter
from dash.core.diversity import (
    get_preliminary_importance, greedy_maxmin_selection,
    cluster_coverage_selection, deduplication_selection,
)
from dash.core.diagnostics import (
    FeatureStabilityIndex, ImportanceStabilityPlot,
    local_disagreement_map, compute_diagnostics,
)
from dash.experiments.synthetic import generate_synthetic_linear, generate_synthetic_nonlinear
from dash.baselines import (
    SingleBestBaseline, LargeSingleModelBaseline, NaiveAveragingBaseline,
    StochasticRetrainBaseline, EnsembleSHAPBaseline, RandomSelectionBaseline,
)
from dash.evaluation import (
    dgp_agreement, importance_accuracy, importance_stability,
    stability_bootstrap_ci, within_group_equity, cohens_d, compare_methods,
    group_level_accuracy, group_level_mse, holm_bonferroni, feature_ablation_score,
)

# Canonical configuration
PAPER_CONFIG = {
    'M': 200, 'K': 30, 'N_REPS': 20, 'EPSILON': 0.08, 'DELTA': 0.05,
    'N_TRIALS_SB': 30, 'T_PER_MODEL': 500, 'N_ESTIMATORS_ESHAP': 2000,
    'TAU_CLUSTER': 0.3,
}
SEED = 42
M = PAPER_CONFIG['M']
K = PAPER_CONFIG['K']
N_REPS = PAPER_CONFIG['N_REPS']
EPSILON = PAPER_CONFIG['EPSILON']
DELTA = PAPER_CONFIG['DELTA']
N_TRIALS_SB = PAPER_CONFIG['N_TRIALS_SB']
feature_names = [f'G{g}_f{j}' for g in range(10) for j in range(5)]
REAL_EPSILON = 0.05
REAL_EPSILON_MODE = 'relative'

print(f'\nDASH loaded. Config: M={M}, K={K}, N_REPS={N_REPS}, \u03b5={EPSILON}, \u03b4={DELTA}')
print(f'  Real-data: \u03b5={REAL_EPSILON} (mode={REAL_EPSILON_MODE})')
_notebook_start = time.time()

# Checkpoint infrastructure
CHECKPOINT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
RESULTS_DIR = os.path.join('results', 'figures')
os.makedirs(RESULTS_DIR, exist_ok=True)

def _ckpt_path(name):
    return os.path.join(CHECKPOINT_DIR, f'ckpt_{name}.pkl')

def save_checkpoint(name, **data):
    path = _ckpt_path(name)
    with open(path, 'wb') as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  [CHECKPOINT] Saved {name} ({size_mb:.1f} MB)')

def load_checkpoint(name):
    path = _ckpt_path(name)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  [CHECKPOINT] Loaded {name} ({size_mb:.1f} MB)')
        return data
    return None

def has_checkpoint(name):
    return os.path.exists(_ckpt_path(name))

def clear_checkpoint(name):
    path = _ckpt_path(name)
    if os.path.exists(path):
        os.remove(path)
        print(f'  [CHECKPOINT] Cleared {name}')

def clear_all_checkpoints():
    for fn in os.listdir(CHECKPOINT_DIR):
        if fn.startswith('ckpt_') and fn.endswith('.pkl'):
            os.remove(os.path.join(CHECKPOINT_DIR, fn))
    print('  [CHECKPOINT] All checkpoints cleared.')

existing = [fn for fn in os.listdir(CHECKPOINT_DIR) if fn.startswith('ckpt_')]
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
if existing:
    print(f'  Found {len(existing)} existing checkpoint(s)')
else:
    print('  No existing checkpoints. Full run will execute.')

## 1. Proof of Concept
Synthetic linear DGP at \u03c1=0.9, single DASH run with diagnostics.

In [ ]:
ckpt = load_checkpoint('v7_sec1_poc')
if ckpt is None:
    t0 = time.time()
    rho = 0.9
    Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, groups, true_imp, _ = \
        generate_synthetic_linear(N=5000, rho=rho, seed=SEED)

    dash_poc = DASHPipeline(
        M=M, K=K, epsilon=EPSILON, delta=DELTA,
        selection_method='maxmin', n_jobs=-1, seed=SEED, verbose=True,
    )
    dash_poc.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, feature_names=feature_names)

    imp = dash_poc.global_importance_
    rho_acc, mse_acc = dgp_agreement(imp, true_imp)
    stab_vecs = [imp]  # Single run — stability is undefined
    eq = within_group_equity(imp, groups)
    gacc = group_level_accuracy(imp, true_imp, groups)

    print(f'DGP agreement: \u03c1={rho_acc:.4f}, MSE={mse_acc:.6f}')
    print(f'Equity (CV): {eq:.4f}')
    print(f'Group accuracy: {gacc:.4f}')
    elapsed = time.time() - t0
    print(f'Completed in {elapsed:.1f}s')

    save_checkpoint('v7_sec1_poc',
        dash=dash_poc, groups=groups, true_imp=true_imp,
        Xexp=Xexp, rho_acc=rho_acc, eq=eq, gacc=gacc)
else:
    dash_poc = ckpt['dash']
    groups = ckpt['groups']
    true_imp = ckpt['true_imp']
    Xexp = ckpt['Xexp']
    print(f"PoC loaded. DGP agreement: {ckpt['rho_acc']:.4f}, Equity: {ckpt['eq']:.4f}")

### 1.1 Importance-Stability Plot

In [ ]:
fig = dash_poc.plot_importance_stability(groups=groups, annotate_top_k=8)
fsi = dash_poc.get_fsi()
fsi.summary(top_k=10)
plt.show()

### 1.2 Local Disagreement Map

In [ ]:
var_obs = np.mean(dash_poc.variance_matrix_, axis=1)
high_var_idx = np.argmax(var_obs)
fig = local_disagreement_map(
    dash_poc.all_shap_matrices_, high_var_idx,
    feature_names=feature_names, top_k=12,
)
plt.show()

## 2. First-Mover Bias Visualization (Paper Figure 2)
Per-feature importance within a correlated group: demonstrates that SB/LSM
concentrate on an arbitrary feature while DASH distributes proportionally.

In [ ]:
ckpt = load_checkpoint('v7_sec2_fmb')
if ckpt is None:
    t0 = time.time()
    rho = 0.9
    group_features = list(range(5))  # Group 1
    group_labels = [feature_names[i] for i in group_features]
    n_vis_reps = 5

    methods_to_run = ['Single Best', 'Large Single Model', 'DASH (MaxMin)']
    method_importances = {m: [] for m in methods_to_run}

    for rep in range(n_vis_reps):
        rep_seed = SEED + rep
        Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, grps, true_imp, _ = \
            generate_synthetic_linear(N=5000, rho=rho, seed=rep_seed)

        sb = SingleBestBaseline(n_trials=N_TRIALS_SB, seed=rep_seed)
        sb.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, seed=rep_seed)
        method_importances['Single Best'].append(sb.global_importance_[group_features])

        lsm = LargeSingleModelBaseline(
            K=K, T_per_model=PAPER_CONFIG['T_PER_MODEL'],
            colsample_bytree=0.2, seed=rep_seed,
        )
        lsm.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, seed=rep_seed)
        method_importances['Large Single Model'].append(lsm.global_importance_[group_features])

        dash_m = DASHPipeline(
            M=M, K=K, epsilon=EPSILON, delta=DELTA,
            selection_method='maxmin', n_jobs=-1, seed=rep_seed, verbose=False,
        )
        dash_m.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, feature_names=feature_names)
        method_importances['DASH (MaxMin)'].append(dash_m.global_importance_[group_features])
        print(f'  Rep {rep+1}/{n_vis_reps} done')

    avg_imp = {m: np.mean(method_importances[m], axis=0) for m in methods_to_run}
    std_imp = {m: np.std(method_importances[m], axis=0, ddof=1) for m in methods_to_run}
    elapsed = time.time() - t0
    print(f'Completed in {elapsed:.1f}s')
    save_checkpoint('v7_sec2_fmb',
        avg_imp=avg_imp, std_imp=std_imp,
        group_labels=group_labels, group_features=group_features)
else:
    avg_imp = ckpt['avg_imp']
    std_imp = ckpt['std_imp']
    group_labels = ckpt['group_labels']
    group_features = ckpt['group_features']
    print('First-mover visualization loaded.')

# Plot and save
true_per_feature = 2.0 / 5
methods_to_run = list(avg_imp.keys())
COLORS = {'Single Best': '#3498db', 'Large Single Model': '#e74c3c', 'DASH (MaxMin)': '#2ecc71'}
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(group_features))
width = 0.22
for i, m in enumerate(methods_to_run):
    ax.bar(x + i * width, avg_imp[m], width, yerr=std_imp[m],
           label=m, color=COLORS.get(m, '#333'), capsize=3, alpha=0.85)
ax.axhline(y=true_per_feature, color='black', linestyle='--', linewidth=1,
           label=f'True importance ({true_per_feature:.2f})', alpha=0.6)
ax.set_xlabel('Feature (within correlated group 1)')
ax.set_ylabel('Global importance (mean |SHAP|)')
ax.set_title(f'First-Mover Bias: Importance Concentration at \u03c1=0.9')
ax.set_xticks(x + width)
ax.set_xticklabels(group_labels)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/first_mover_concentration.pdf', bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/first_mover_concentration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/first_mover_concentration.pdf')

## 3. Correlation Sweep — Main Result (Paper Table 1 & Figure 3)
Full sweep \u03c1 \u2208 {0.0, 0.5, 0.7, 0.9, 0.95} with 8 methods, N_REPS=20.
This is the central experiment.

In [ ]:
ckpt = load_checkpoint('v7_sec3_sweep')
if ckpt is None:
    t0 = time.time()
    RHO_LEVELS = [0.0, 0.5, 0.7, 0.9, 0.95]
    sweep_results = {}

    for rho in RHO_LEVELS:
        print(f'\n{"="*60}\n  Correlation sweep: \u03c1 = {rho}\n{"="*60}')
        sweep_results[rho] = {}

        method_specs = [
            ('Single Best', lambda s: SingleBestBaseline(n_trials=N_TRIALS_SB, seed=s)),
            ('Single Best (M)', lambda s: SingleBestBaseline(n_trials=M, seed=s)),
            ('Large Single Model', lambda s: LargeSingleModelBaseline(
                K=K, T_per_model=PAPER_CONFIG['T_PER_MODEL'],
                colsample_bytree=0.2, seed=s)),
            ('LSM (Tuned)', lambda s: LargeSingleModelBaseline(
                K=K, T_per_model=PAPER_CONFIG['T_PER_MODEL'],
                colsample_bytree=0.2, tune=True, seed=s)),
            ('Stoch. Retrain', lambda s: StochasticRetrainBaseline(
                N=K, task='regression', n_jobs=-1, seed=s)),
            ('Random Selection', lambda s: RandomSelectionBaseline(
                M=M, K=K, epsilon=EPSILON, delta=DELTA, n_jobs=-1, seed=s)),
            ('DASH (MaxMin)', lambda s: DASHPipeline(
                M=M, K=K, epsilon=EPSILON, delta=DELTA,
                selection_method='maxmin', n_jobs=-1, seed=s, verbose=False)),
        ]

        for method_name, method_fn in method_specs:
            t_m = time.time()
            imp_runs, acc_runs, eq_runs, rmse_runs, keff_runs = [], [], [], [], []

            for rep in range(N_REPS):
                rep_seed = SEED + rep
                Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, grps, true_imp, _ = \
                    generate_synthetic_linear(N=5000, rho=rho, seed=rep_seed)

                m = method_fn(rep_seed)
                if hasattr(m, 'fit_from_population'):
                    # NaiveAveraging needs a pre-built population — skip for sweep
                    continue
                m.fit(Xtr, ytr, Xv, yv, X_ref=Xexp,
                      **(dict(feature_names=feature_names) if isinstance(m, DASHPipeline) else {}),
                      **(dict(seed=rep_seed) if not isinstance(m, DASHPipeline) else {}))
                imp = m.global_importance_
                r, _ = dgp_agreement(imp, true_imp)
                eq = within_group_equity(imp, grps)
                # RMSE
                if hasattr(m, 'get_consensus_ensemble_predictions'):
                    preds = m.get_consensus_ensemble_predictions(Xte)
                elif hasattr(m, 'model_'):
                    preds = m.model_.predict(Xte)
                else:
                    preds = None
                rmse = float(np.sqrt(np.mean((yte - preds)**2))) if preds is not None else np.nan

                imp_runs.append(imp)
                acc_runs.append(r)
                eq_runs.append(eq)
                rmse_runs.append(rmse)
                if hasattr(m, 'selected_indices_') and m.selected_indices_ is not None:
                    keff_runs.append(len(m.selected_indices_))

            if not imp_runs:
                continue

            stab, stab_se, stab_ci_lo, stab_ci_hi = stability_bootstrap_ci(imp_runs)
            elapsed_m = time.time() - t_m

            sweep_results[rho][method_name] = {
                'stability': stab, 'stability_se': stab_se,
                'stability_ci_lo': stab_ci_lo, 'stability_ci_hi': stab_ci_hi,
                'accuracy_mean': float(np.mean(acc_runs)),
                'accuracy_std': float(np.std(acc_runs, ddof=1)),
                'equity_mean': float(np.mean(eq_runs)),
                'equity_std': float(np.std(eq_runs, ddof=1)),
                'rmse_mean': float(np.nanmean(rmse_runs)),
                'rmse_std': float(np.nanstd(rmse_runs, ddof=1)),
                'k_eff_mean': float(np.mean(keff_runs)) if keff_runs else None,
                'elapsed_s': elapsed_m,
                'acc_runs': np.array(acc_runs),
                'eq_runs': np.array(eq_runs),
                'rmse_runs': np.array(rmse_runs),
                'imp_runs': imp_runs,
            }
            print(f'  {method_name:<24} stab={stab:.4f}\u00b1{stab_se:.4f}  '
                  f'acc={np.mean(acc_runs):.4f}  eq={np.mean(eq_runs):.4f}  '
                  f'rmse={np.nanmean(rmse_runs):.4f}  [{elapsed_m:.0f}s]')

    elapsed = time.time() - t0
    print(f'\nSweep completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec3_sweep', sweep_results=sweep_results)
else:
    sweep_results = ckpt['sweep_results']
    print('Correlation sweep loaded.')

In [ ]:
print('\n' + '='*90)
print('TABLE 1: Correlation Sweep (Paper Table 1)')
print('='*90)
print(f'{"rho":<6} {"Method":<24} {"Stability":>14} {"Accuracy":>10} {"Equity":>10} {"RMSE":>10} {"Time(s)":>10}')
print('-'*90)
for rho in sorted(sweep_results.keys()):
    for method, data in sweep_results[rho].items():
        stab = data['stability']
        se = data.get('stability_se', 0)
        acc = data.get('accuracy_mean', 0)
        eq = data.get('equity_mean', 0)
        rmse = data.get('rmse_mean', 0)
        elapsed = data.get('elapsed_s', 0)
        print(f'{rho:<6.2f} {method:<24} {stab:.4f}\u00b1{se:.4f}  {acc:>10.4f} {eq:>10.4f} {rmse:>10.4f} {elapsed:>10.0f}')
    print()

In [ ]:
RHO_LEVELS = sorted(sweep_results.keys())
COLORS = {
    'Single Best': '#3498db', 'Single Best (M)': '#2471a3',
    'Large Single Model': '#e74c3c', 'LSM (Tuned)': '#c0392b',
    'Stoch. Retrain': '#f39c12', 'Random Selection': '#9b59b6',
    'DASH (MaxMin)': '#2ecc71', 'Naive Top-N': '#1abc9c',
}
MARKERS = {
    'Single Best': 's', 'Single Best (M)': 'S',
    'Large Single Model': 'X', 'LSM (Tuned)': 'x',
    'Stoch. Retrain': 'D', 'Random Selection': '^',
    'DASH (MaxMin)': 'o', 'Naive Top-N': 'v',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
method_names = list(sweep_results[RHO_LEVELS[0]].keys())

for name in method_names:
    c = COLORS.get(name, '#333')
    m = MARKERS.get(name, 'o')

    vals = [sweep_results[rho][name]['stability'] for rho in RHO_LEVELS if name in sweep_results[rho]]
    rhos = [rho for rho in RHO_LEVELS if name in sweep_results[rho]]
    axes[0].plot(rhos, vals, f'{m}-', color=c, label=name, linewidth=2, markersize=7)

    vals = [sweep_results[rho][name]['accuracy_mean'] for rho in rhos]
    axes[1].plot(rhos, vals, f'{m}-', color=c, label=name, linewidth=2, markersize=7)

    vals = [sweep_results[rho][name]['equity_mean'] for rho in rhos]
    axes[2].plot(rhos, vals, f'{m}-', color=c, label=name, linewidth=2, markersize=7)

axes[0].set_ylabel('Stability (Spearman)')
axes[0].set_title('Stability vs. Collinearity')
axes[0].legend(fontsize=7, loc='lower left')
axes[1].set_ylabel('Accuracy (Spearman vs DGP)')
axes[1].set_title('Accuracy vs. Collinearity')
axes[2].set_ylabel('Within-Group CV (lower=better)')
axes[2].set_title('Equity vs. Collinearity')
for ax in axes:
    ax.set_xlabel('Within-Group Correlation \u03c1')

fig.suptitle('DASH vs Baselines \u2014 Synthetic Linear DGP', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/correlation_sweep.pdf', bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/correlation_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/correlation_sweep.pdf')

## 4. Extended Baselines at \u03c1=0.9 (Paper Table 2)
Run Ensemble SHAP and Naive Top-N to complete the baseline taxonomy.

In [ ]:
ckpt = load_checkpoint('v7_sec4_table2')
if ckpt is None:
    t0 = time.time()
    rho = 0.9
    extra_results = {}

    # Ensemble SHAP
    imp_runs_es, acc_runs_es, eq_runs_es = [], [], []
    for rep in range(N_REPS):
        rep_seed = SEED + rep
        Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, grps, true_imp, _ = \
            generate_synthetic_linear(N=5000, rho=rho, seed=rep_seed)
        es = EnsembleSHAPBaseline(n_estimators=PAPER_CONFIG['N_ESTIMATORS_ESHAP'], seed=rep_seed)
        es.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, seed=rep_seed)
        imp_runs_es.append(es.global_importance_)
        r, _ = dgp_agreement(es.global_importance_, true_imp)
        acc_runs_es.append(r)
        eq_runs_es.append(within_group_equity(es.global_importance_, grps))
    stab, se, _, _ = stability_bootstrap_ci(imp_runs_es)
    extra_results['Ensemble SHAP'] = {
        'stability': stab, 'stability_se': se,
        'accuracy_mean': float(np.mean(acc_runs_es)),
        'equity_mean': float(np.mean(eq_runs_es)),
    }
    print(f'Ensemble SHAP: stab={stab:.4f}, acc={np.mean(acc_runs_es):.4f}')

    # Naive Top-N (uses DASH population from sweep)
    if 0.9 in sweep_results and 'DASH (MaxMin)' in sweep_results[0.9]:
        imp_runs_naive, acc_runs_naive, eq_runs_naive = [], [], []
        for rep in range(N_REPS):
            rep_seed = SEED + rep
            Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, grps, true_imp, _ = \
                generate_synthetic_linear(N=5000, rho=rho, seed=rep_seed)
            # Train DASH population, then use NaiveAveraging on it
            dm = DASHPipeline(
                M=M, K=K, epsilon=EPSILON, delta=DELTA,
                selection_method='maxmin', n_jobs=-1, seed=rep_seed, verbose=False)
            dm.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, feature_names=feature_names)
            na = NaiveAveragingBaseline(N=K)
            na.fit_from_population(dm.models_, dm.val_scores_, Xexp)
            imp_runs_naive.append(na.global_importance_)
            r, _ = dgp_agreement(na.global_importance_, true_imp)
            acc_runs_naive.append(r)
            eq_runs_naive.append(within_group_equity(na.global_importance_, grps))
        stab, se, _, _ = stability_bootstrap_ci(imp_runs_naive)
        extra_results['Naive Top-N'] = {
            'stability': stab, 'stability_se': se,
            'accuracy_mean': float(np.mean(acc_runs_naive)),
            'equity_mean': float(np.mean(eq_runs_naive)),
        }
        print(f'Naive Top-N: stab={stab:.4f}, acc={np.mean(acc_runs_naive):.4f}')

    elapsed = time.time() - t0
    print(f'Table 2 extras completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec4_table2', extra_results=extra_results)
else:
    extra_results = ckpt['extra_results']
    print('Table 2 extras loaded.')

In [ ]:
print('\n' + '='*80)
print('TABLE 2: All Methods at \u03c1=0.9 (Paper Table 2)')
print('='*80)

# Merge sweep results at rho=0.9 with extra baselines
all_09 = {}
if 0.9 in sweep_results:
    all_09.update(sweep_results[0.9])
all_09.update(extra_results)

print(f'{"Method":<24} {"Stability":>14} {"Accuracy":>10} {"Equity":>10}')
print('-'*60)
for method, data in all_09.items():
    stab = data['stability']
    se = data.get('stability_se', 0)
    acc = data.get('accuracy_mean', 0)
    eq = data.get('equity_mean', 0)
    print(f'{method:<24} {stab:.4f}\u00b1{se:.4f}  {acc:>10.4f} {eq:>10.4f}')

# Significance tests: DASH vs each baseline
print('\nSignificance tests (DASH MaxMin vs baselines):')
dash_data = all_09.get('DASH (MaxMin)', {})
if 'imp_runs' in dash_data:
    for bname, bdata in all_09.items():
        if bname == 'DASH (MaxMin)' or 'imp_runs' not in bdata:
            continue
        # Stability comparison via per-rep pairwise correlations
        d = cohens_d(
            [importance_stability([dash_data['imp_runs'][i], dash_data['imp_runs'][(i+1) % N_REPS]]) for i in range(N_REPS)],
            [importance_stability([bdata['imp_runs'][i], bdata['imp_runs'][(i+1) % N_REPS]]) for i in range(N_REPS)]
        )
        print(f'  vs {bname:<22} Cohen\'s d = {d:+.3f}')

## 5. Real-World: Breast Cancer (Paper Table 4, Figures 4-5)
Binary classification with heavy natural collinearity (21 pairs |r|>0.9).

In [ ]:
ckpt = load_checkpoint('v7_sec5_bc')
if ckpt is None:
    t0 = time.time()
    from sklearn.datasets import load_breast_cancer
    bc = load_breast_cancer()
    X_bc, y_bc = bc.data, bc.target
    bc_names = list(bc.feature_names)

    X_bc_pool, X_bc_test, y_bc_pool, y_bc_test = train_test_split(
        X_bc, y_bc, test_size=0.2, random_state=SEED)

    bc_results = {}
    for name in ['Single Best', 'DASH (MaxMin)']:
        imp_runs, ablation_runs = [], []
        for rep in range(N_REPS):
            rep_seed = SEED + rep
            Xtr_r, Xv_r, ytr_r, yv_r = train_test_split(
                X_bc_pool, y_bc_pool, test_size=0.2, random_state=rep_seed)
            Xtr_r, Xexp_r, ytr_r, yexp_r = train_test_split(
                Xtr_r, ytr_r, test_size=0.12, random_state=rep_seed)
            scaler_r = StandardScaler().fit(Xtr_r)
            Xtr_r = scaler_r.transform(Xtr_r)
            Xv_r = scaler_r.transform(Xv_r)
            Xexp_r = scaler_r.transform(Xexp_r)
            Xte_r = scaler_r.transform(X_bc_test)

            if name == 'Single Best':
                m = SingleBestBaseline(n_trials=N_TRIALS_SB, task='binary', seed=rep_seed)
                m.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=Xexp_r)
                imp = m.global_importance_
                abl = feature_ablation_score(m.model_, Xte_r, y_bc_test, imp)
            else:
                m = DASHPipeline(
                    M=M, K=K, epsilon=EPSILON, delta=DELTA,
                    selection_method='maxmin', task='binary',
                    n_jobs=-1, seed=rep_seed, verbose=False)
                m.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=Xexp_r, feature_names=bc_names)
                imp = m.global_importance_
                abl = feature_ablation_score(m.selected_models_[0], Xte_r, y_bc_test, imp)
            imp_runs.append(imp)
            ablation_runs.append(abl)

        stab, se, ci_lo, ci_hi = stability_bootstrap_ci(imp_runs)
        bc_results[name] = {
            'stability': stab, 'stability_se': se,
            'stability_ci_lo': ci_lo, 'stability_ci_hi': ci_hi,
            'ablation_mean': float(np.mean(ablation_runs)),
            'ablation_std': float(np.std(ablation_runs, ddof=1)),
        }
        print(f'  {name:<22} stab={stab:.4f}\u00b1{se:.4f}  abl={np.mean(ablation_runs):.4f}')

    elapsed = time.time() - t0
    print(f'Breast Cancer completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec5_bc',
        bc_results=bc_results, last_dash=m, bc_names=bc_names)
else:
    bc_results = ckpt['bc_results']
    m = ckpt['last_dash']
    bc_names = ckpt['bc_names']
    print('Breast Cancer loaded.')
    for name, data in bc_results.items():
        print(f'  {name}: stab={data["stability"]:.4f}')

In [ ]:
fig = m.plot_importance_stability(title='IS Plot \u2014 Breast Cancer', annotate_top_k=8)
fig.savefig(f'{RESULTS_DIR}/is_plot_breast_cancer.png', dpi=150, bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/is_plot_breast_cancer.pdf', bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/is_plot_breast_cancer.pdf')

In [ ]:
var_obs = np.mean(m.variance_matrix_, axis=1)
fig = local_disagreement_map(
    m.all_shap_matrices_, np.argmax(var_obs),
    feature_names=bc_names, top_k=12)
fig.savefig(f'{RESULTS_DIR}/disagreement_breast_cancer.png', dpi=150, bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/disagreement_breast_cancer.pdf', bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/disagreement_breast_cancer.pdf')

## 6. Real-World: Superconductor (Paper Table 4)

In [ ]:
ckpt = load_checkpoint('v7_sec6_sc')
if ckpt is None:
    t0 = time.time()
    try:
        from sklearn.datasets import fetch_openml
        sc = fetch_openml('superconduct', version=1, as_frame=False)
        X_sc, y_sc = sc.data, sc.target.astype(float)
    except Exception as e:
        print(f'Could not fetch Superconductor dataset: {e}')
        print('Skipping this experiment.')
        sc_results = {'_skipped': True}
        save_checkpoint('v7_sec6_sc', sc_results=sc_results)
        raise SystemExit('Skipped SC')

    X_sc_pool, X_sc_test, y_sc_pool, y_sc_test = train_test_split(
        X_sc, y_sc, test_size=0.2, random_state=SEED)

    sc_results = {}
    for name in ['Single Best', 'Large Single Model', 'DASH (MaxMin)']:
        imp_runs, rmse_runs = [], []
        for rep in range(N_REPS):
            rep_seed = SEED + rep
            Xtr_r, Xv_r, ytr_r, yv_r = train_test_split(
                X_sc_pool, y_sc_pool, test_size=0.2, random_state=rep_seed)
            Xtr_r, Xexp_r, ytr_r, yexp_r = train_test_split(
                Xtr_r, ytr_r, test_size=0.12, random_state=rep_seed)
            scaler_r = StandardScaler().fit(Xtr_r)
            Xtr_r, Xv_r = scaler_r.transform(Xtr_r), scaler_r.transform(Xv_r)
            Xexp_r, Xte_r = scaler_r.transform(Xexp_r), scaler_r.transform(X_sc_test)

            if name == 'Single Best':
                mod = SingleBestBaseline(n_trials=N_TRIALS_SB, seed=rep_seed)
                mod.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=Xexp_r, seed=rep_seed)
                preds = mod.model_.predict(Xte_r)
            elif name == 'Large Single Model':
                mod = LargeSingleModelBaseline(
                    K=K, T_per_model=PAPER_CONFIG['T_PER_MODEL'],
                    colsample_bytree=0.2, seed=rep_seed)
                mod.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=Xexp_r, seed=rep_seed)
                preds = mod.model_.predict(Xte_r)
            else:
                mod = DASHPipeline(
                    M=M, K=K, epsilon=REAL_EPSILON, delta=DELTA,
                    epsilon_mode=REAL_EPSILON_MODE,
                    selection_method='maxmin', n_jobs=-1, seed=rep_seed, verbose=False)
                mod.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=Xexp_r)
                preds = mod.get_consensus_ensemble_predictions(Xte_r)
            imp_runs.append(mod.global_importance_)
            rmse_runs.append(float(np.sqrt(np.mean((y_sc_test - preds)**2))))

        stab, se, _, _ = stability_bootstrap_ci(imp_runs)
        sc_results[name] = {
            'stability': stab, 'stability_se': se,
            'rmse_mean': float(np.mean(rmse_runs)),
            'rmse_std': float(np.std(rmse_runs, ddof=1)),
        }
        print(f'  {name:<22} stab={stab:.4f}  rmse={np.mean(rmse_runs):.2f}')

    elapsed = time.time() - t0
    print(f'Superconductor completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec6_sc', sc_results=sc_results)
else:
    sc_results = ckpt['sc_results']
    print('Superconductor loaded.')
    for name, data in sc_results.items():
        if name.startswith('_'): continue
        print(f'  {name}: stab={data["stability"]:.4f}, rmse={data["rmse_mean"]:.2f}')

## 7. Real-World: California Housing (Paper Table 4)

In [ ]:
ckpt = load_checkpoint('v7_sec7_california')
if ckpt is None:
    t0 = time.time()
    from sklearn.datasets import fetch_california_housing
    cal = fetch_california_housing()
    X_cal, y_cal = cal.data, cal.target
    cal_names = list(cal.feature_names)

    X_cal_pool, X_cal_test, y_cal_pool, y_cal_test = train_test_split(
        X_cal, y_cal, test_size=0.2, random_state=SEED)

    cal_results = {}
    last_dash_cal = None
    for name in ['Single Best', 'DASH (MaxMin)']:
        imp_runs, rmse_runs = [], []
        for rep in range(N_REPS):
            rep_seed = SEED + rep
            Xtr_r, Xv_r, ytr_r, yv_r = train_test_split(
                X_cal_pool, y_cal_pool, test_size=0.2, random_state=rep_seed)
            Xtr_r, Xexp_r, ytr_r, yexp_r = train_test_split(
                Xtr_r, ytr_r, test_size=0.12, random_state=rep_seed)
            scaler_r = StandardScaler().fit(Xtr_r)
            Xtr_r, Xv_r = scaler_r.transform(Xtr_r), scaler_r.transform(Xv_r)
            Xexp_r, Xte_r = scaler_r.transform(Xexp_r), scaler_r.transform(X_cal_test)

            if name == 'Single Best':
                mod = SingleBestBaseline(n_trials=N_TRIALS_SB, seed=rep_seed)
                mod.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=Xexp_r, seed=rep_seed)
                preds = mod.model_.predict(Xte_r)
            else:
                mod = DASHPipeline(
                    M=M, K=K, epsilon=REAL_EPSILON, delta=DELTA,
                    epsilon_mode=REAL_EPSILON_MODE,
                    selection_method='maxmin', n_jobs=-1, seed=rep_seed, verbose=False)
                mod.fit(Xtr_r, ytr_r, Xv_r, yv_r, X_ref=Xexp_r, feature_names=cal_names)
                preds = mod.get_consensus_ensemble_predictions(Xte_r)
                last_dash_cal = mod
            imp_runs.append(mod.global_importance_)
            rmse_runs.append(float(np.sqrt(np.mean((y_cal_test - preds)**2))))

        stab, se, _, _ = stability_bootstrap_ci(imp_runs)
        cal_results[name] = {
            'stability': stab, 'stability_se': se,
            'rmse_mean': float(np.mean(rmse_runs)),
            'rmse_std': float(np.std(rmse_runs, ddof=1)),
        }
        print(f'  {name:<22} stab={stab:.4f}  rmse={np.mean(rmse_runs):.4f}')

    elapsed = time.time() - t0
    print(f'California Housing completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec7_california',
        cal_results=cal_results, last_dash=last_dash_cal, cal_names=cal_names)
else:
    cal_results = ckpt['cal_results']
    last_dash_cal = ckpt.get('last_dash')
    cal_names = ckpt.get('cal_names')
    print('California Housing loaded.')
    for name, data in cal_results.items():
        print(f'  {name}: stab={data["stability"]:.4f}, rmse={data["rmse_mean"]:.4f}')

In [ ]:
if last_dash_cal is not None:
    fig = last_dash_cal.plot_importance_stability(
        title='IS Plot \u2014 California Housing', annotate_top_k=8)
    fig.savefig(f'{RESULTS_DIR}/is_plot_california.png', dpi=150, bbox_inches='tight')
    fig.savefig(f'{RESULTS_DIR}/is_plot_california.pdf', bbox_inches='tight')
    plt.show()
    print(f'Saved: {RESULTS_DIR}/is_plot_california.pdf')
else:
    print('No DASH model available for IS plot.')

## 8. Epsilon Sensitivity (Paper Table 5)

In [ ]:
ckpt = load_checkpoint('v7_sec8_epsilon')
if ckpt is None:
    t0 = time.time()
    rho = 0.9
    eps_values = [0.03, 0.05, 0.08, 0.10]
    eps_results = {}

    for eps_val in eps_values:
        imp_runs, acc_runs, keff_runs, n_passing_runs = [], [], [], []
        for rep in range(N_REPS):
            rep_seed = SEED + rep
            Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, grps, true_imp, _ = \
                generate_synthetic_linear(N=5000, rho=rho, seed=rep_seed)
            dm = DASHPipeline(
                M=M, K=K, epsilon=eps_val, delta=DELTA,
                selection_method='maxmin', n_jobs=-1, seed=rep_seed, verbose=False)
            dm.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, feature_names=feature_names)
            imp_runs.append(dm.global_importance_)
            r, _ = dgp_agreement(dm.global_importance_, true_imp)
            acc_runs.append(r)
            keff_runs.append(len(dm.selected_indices_))
            n_passing_runs.append(len(dm.filtered_indices_))

        stab, se, _, _ = stability_bootstrap_ci(imp_runs)
        eps_results[eps_val] = {
            'stability': stab, 'stability_se': se,
            'accuracy_mean': float(np.mean(acc_runs)),
            'n_passing': float(np.mean(n_passing_runs)),
            'k_eff_mean': float(np.mean(keff_runs)),
            'k_eff_std': float(np.std(keff_runs, ddof=1)),
        }
        print(f'  \u03b5={eps_val:.2f}  passing={np.mean(n_passing_runs):.0f}  '
              f'K_eff={np.mean(keff_runs):.1f}  stab={stab:.4f}  acc={np.mean(acc_runs):.4f}')

    elapsed = time.time() - t0
    print(f'Epsilon sensitivity completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec8_epsilon', eps_results=eps_results)
else:
    eps_results = ckpt['eps_results']
    print('Epsilon sensitivity loaded.')

In [ ]:
print('\nTABLE 5: Epsilon Sensitivity')
print(f'{"eps":>6} {"Passing":>10} {"K_eff":>12} {"Stability":>14} {"Accuracy":>10}')
print('-'*55)
for eps_val, data in sorted(eps_results.items()):
    print(f'{eps_val:>6.2f} {data["n_passing"]:>10.1f} '
          f'{data["k_eff_mean"]:>5.1f}\u00b1{data.get("k_eff_std",0):>4.1f}  '
          f'{data["stability"]:.4f}\u00b1{data.get("stability_se",0):.4f}  '
          f'{data["accuracy_mean"]:>10.4f}')

## 9. Nonlinear DGP (Paper Table 6)

In [ ]:
ckpt = load_checkpoint('v7_sec9_nonlinear')
if ckpt is None:
    t0 = time.time()
    RHO_LEVELS = [0.0, 0.5, 0.7, 0.9, 0.95]
    nl_results = {}

    for rho in RHO_LEVELS:
        print(f'  Nonlinear sweep: \u03c1={rho}')
        nl_results[rho] = {}
        for method_name, method_fn in [
            ('Single Best', lambda s: SingleBestBaseline(n_trials=N_TRIALS_SB, seed=s)),
            ('DASH (MaxMin)', lambda s: DASHPipeline(
                M=M, K=K, epsilon=EPSILON, delta=DELTA,
                selection_method='maxmin', n_jobs=-1, seed=s, verbose=False)),
        ]:
            imp_runs, eq_runs = [], []
            for rep in range(N_REPS):
                rep_seed = SEED + rep
                Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, grps, true_imp, _ = \
                    generate_synthetic_nonlinear(N=5000, rho=rho, seed=rep_seed)
                mod = method_fn(rep_seed)
                if isinstance(mod, DASHPipeline):
                    mod.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, feature_names=feature_names)
                else:
                    mod.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, seed=rep_seed)
                imp_runs.append(mod.global_importance_)
                eq_runs.append(within_group_equity(mod.global_importance_, grps))
            stab, se, _, _ = stability_bootstrap_ci(imp_runs)
            nl_results[rho][method_name] = {
                'stability': stab, 'stability_se': se,
                'equity_mean': float(np.mean(eq_runs)),
            }
            print(f'    {method_name:<22} stab={stab:.4f}  eq={np.mean(eq_runs):.4f}')

    elapsed = time.time() - t0
    print(f'Nonlinear sweep completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec9_nonlinear', nl_results=nl_results)
else:
    nl_results = ckpt['nl_results']
    print('Nonlinear sweep loaded.')

In [ ]:
print('\nTABLE 6: Nonlinear DGP')
print(f'{"rho":<6} {"SB Stab":>10} {"DASH Stab":>12} {"SB Equity":>10} {"DASH Equity":>12}')
print('-'*55)
for rho in sorted(nl_results.keys()):
    sb = nl_results[rho].get('Single Best', {})
    da = nl_results[rho].get('DASH (MaxMin)', {})
    print(f'{rho:<6.2f} {sb.get("stability",0):>10.4f} {da.get("stability",0):>12.4f} '
          f'{sb.get("equity_mean",0):>10.4f} {da.get("equity_mean",0):>12.4f}')

## 10. Ablation Studies (Paper Figure 6)

In [ ]:
ckpt = load_checkpoint('v7_sec10_ablation')
if ckpt is None:
    t0 = time.time()
    ABL_RHOS = [0.0, 0.9, 0.95]
    ABL_DEFAULTS = {'M': M, 'K': K, 'eps': EPSILON, 'delta': DELTA}
    ablations = {
        'M': [50, 100, 200, 500],
        'K': [5, 10, 20, 30, 50],
        'epsilon': [0.01, 0.03, 0.05, 0.08, 0.10],
        'delta': [0.01, 0.05, 0.10, 0.20],
    }
    param_map = {'M': 'M', 'K': 'K', 'epsilon': 'eps', 'delta': 'delta'}

    abl_results = {rho: {} for rho in ABL_RHOS}
    for abl_rho in ABL_RHOS:
        print(f'\n  Ablation at \u03c1={abl_rho}')
        for param_name, values in ablations.items():
            abl_results[abl_rho][param_name] = {}
            for val in values:
                p = ABL_DEFAULTS.copy()
                p[param_map[param_name]] = val
                imp_runs = []
                for rep in range(N_REPS):
                    rep_seed = SEED + rep
                    Xtr, ytr, Xv, yv, Xexp, yexp, Xte, yte, grps, true_imp, _ = \
                        generate_synthetic_linear(N=5000, rho=abl_rho, seed=rep_seed)
                    dm = DASHPipeline(
                        M=p['M'], K=p['K'], epsilon=p['eps'], delta=p['delta'],
                        selection_method='maxmin', n_jobs=-1, seed=rep_seed, verbose=False)
                    dm.fit(Xtr, ytr, Xv, yv, X_ref=Xexp, feature_names=feature_names)
                    imp_runs.append(dm.global_importance_)
                stab = importance_stability(imp_runs)
                abl_results[abl_rho][param_name][val] = {'stability': stab}
                print(f'    {param_name}={val}  stab={stab:.4f}')

    elapsed = time.time() - t0
    print(f'Ablation completed in {elapsed/60:.1f} min')
    save_checkpoint('v7_sec10_ablation', abl_results=abl_results)
else:
    abl_results = ckpt['abl_results']
    print('Ablation loaded.')

In [ ]:
param_configs = [
    ('M', [50, 100, 200, 500], 'Population Size $M$'),
    ('K', [5, 10, 20, 30, 50], 'Selected Models $K$'),
    ('epsilon', [0.01, 0.03, 0.05, 0.08, 0.10], 'Filter Threshold $\\varepsilon$'),
    ('delta', [0.01, 0.05, 0.10, 0.20], 'Diversity Threshold $\\delta$'),
]
rho_styles = {
    0.0:  ('--', '#7f8c8d', 'o', '$\\rho=0.0$'),
    0.9:  ('-',  '#2980b9', 's', '$\\rho=0.9$'),
    0.95: ('-',  '#e74c3c', '^', '$\\rho=0.95$'),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes_flat = axes.flatten()

for ax_idx, (param_name, param_vals, xlabel) in enumerate(param_configs):
    ax = axes_flat[ax_idx]
    for rho_val, (ls, color, marker, label) in rho_styles.items():
        param_data = abl_results.get(rho_val, {}).get(param_name, {})
        x_vals, y_vals = [], []
        for v in param_vals:
            if v in param_data:
                x_vals.append(v)
                y_vals.append(param_data[v].get('stability', 0))
        if x_vals:
            ax.plot(x_vals, y_vals, f'{marker}{ls}', color=color,
                    label=label, linewidth=2, markersize=6)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Stability')
    if ax_idx == 0:
        ax.legend(fontsize=8)

fig.suptitle('DASH Hyperparameter Sensitivity', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/ablation_sensitivity.pdf', bbox_inches='tight')
fig.savefig(f'{RESULTS_DIR}/ablation_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/ablation_sensitivity.pdf')

## 11. Computational Cost Summary (Paper Table)

In [ ]:
print('\n' + '='*70)
print('COMPUTATIONAL COST: Wall-clock timings at \u03c1=0.9')
print('='*70)
if 0.9 in sweep_results:
    print(f'{"Method":<24} {"Total (s)":>10} {"Per-rep (s)":>12}')
    print('-'*48)
    for name, data in sweep_results[0.9].items():
        elapsed = data.get('elapsed_s', 0)
        per_rep = elapsed / N_REPS if N_REPS > 0 else 0
        print(f'{name:<24} {elapsed:>10.1f} {per_rep:>12.1f}')
else:
    print('No sweep results at \u03c1=0.9 available.')

## 12. Success Criteria

In [ ]:
print('\n' + '='*70)
print('SUCCESS CRITERIA')
print('='*70)
criteria = []

# C1: Stability wins (linear) — DASH > SB on >=4/5 rho levels
wins = sum(1 for rho in sweep_results
           if 'DASH (MaxMin)' in sweep_results[rho] and 'Single Best' in sweep_results[rho]
           and sweep_results[rho]['DASH (MaxMin)']['stability'] > sweep_results[rho]['Single Best']['stability'])
total_rho = sum(1 for rho in sweep_results
                if 'DASH (MaxMin)' in sweep_results[rho] and 'Single Best' in sweep_results[rho])
passed = wins >= 4
criteria.append(('Stability wins (linear)', f'{wins}/{total_rho}', passed))
print(f'1. Stability wins: {wins}/{total_rho} \u2192 {"PASS" if passed else "FAIL"}')

# C2: Accuracy at rho=0.9
if 0.9 in sweep_results and 'DASH (MaxMin)' in sweep_results[0.9]:
    acc = sweep_results[0.9]['DASH (MaxMin)']['accuracy_mean']
    passed = acc >= 0.90
    criteria.append(('Accuracy at \u03c1=0.9', f'{acc:.4f}', passed))
    print(f'2. Accuracy at \u03c1=0.9: {acc:.4f} \u2192 {"PASS" if passed else "FAIL"}')

# C3: Safety at rho=0
if 0.0 in sweep_results and 'DASH (MaxMin)' in sweep_results[0.0] and 'Single Best' in sweep_results[0.0]:
    gap = abs(sweep_results[0.0]['DASH (MaxMin)']['stability'] - sweep_results[0.0]['Single Best']['stability'])
    passed = gap < 0.1
    criteria.append(('Safety at \u03c1=0', f'gap={gap:.4f}', passed))
    print(f'3. Safety at \u03c1=0: gap={gap:.4f} \u2192 {"PASS" if passed else "FAIL"}')

# C4: BC stability
if bc_results and 'DASH (MaxMin)' in bc_results:
    bc_stab = bc_results['DASH (MaxMin)']['stability']
    passed = bc_stab > 0.80
    criteria.append(('Breast Cancer stab>0.80', f'{bc_stab:.4f}', passed))
    print(f'4. Breast Cancer: {bc_stab:.4f} \u2192 {"PASS" if passed else "FAIL"}')

# Summary
n_pass = sum(1 for _, _, p in criteria if p)
print(f'\nOverall: {n_pass}/{len(criteria)} criteria passed')

elapsed_total = time.time() - _notebook_start
print(f'\nTotal notebook runtime: {elapsed_total/60:.1f} min')